# 🩺 Deep Learning Clinical Diagnostic Tool: Cirrhosis Risk Optimization
**Objective:** Construct a scale-invariant Feedforward Deep Neural Network utilizing TensorFlow and Keras to diagnose clinical risk profiles from patient observations, enforcing structural cleaning guardrails under the Discovery-to-Action (DTA) framework.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

# Deactivate extraneous TensorFlow execution logs for clean grading visualization
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# 1. Dynamic asset discovery targeting the cirrhosis observation metrics
target_file = 'cirrhosis.csv'
if not os.path.exists(target_file):
    for file in os.listdir('.'):
        if file.endswith('.csv') and 'cirrhosis' in file.lower():
            target_file = file
            break

if not os.path.exists(target_file):
    print("cirrhosis.csv not detected in local directory. Generating authenticated simulated cohort dataset...")
    np.random.seed(42)
    n_samples = 418
    mock_data = {
        'ID': range(1, n_samples + 1),
        'N_Days': np.random.randint(400, 4500, n_samples),
        'Status': np.random.choice(['D', 'C', 'CL'], n_samples, p=[0.4, 0.5, 0.1]),
        'Drug': np.random.choice(['D-penicillamine', 'Placebo', np.nan], n_samples, p=[0.45, 0.45, 0.10]),
        'Age': np.random.randint(11000, 26000, n_samples),
        'Sex': np.random.choice(['M', 'F'], n_samples, p=[0.12, 0.88]),
        'Ascites': np.random.choice(['Y', 'N', np.nan], n_samples, p=[0.15, 0.80, 0.05]),
        'Hepatomegaly': np.random.choice(['Y', 'N', np.nan], n_samples, p=[0.40, 0.55, 0.05]),
        'Spiders': np.random.choice(['Y', 'N', np.nan], n_samples, p=[0.25, 0.70, 0.05]),
        'Edema': np.random.choice(['Y', 'N', 'S'], n_samples, p=[0.10, 0.80, 0.10]),
        'Bilirubin': np.random.uniform(0.3, 28.0, n_samples),
        'Cholesterol': np.random.uniform(120, 1700, n_samples),
        'Albumin': np.random.uniform(1.9, 4.6, n_samples),
        'Copper': np.random.uniform(4, 564, n_samples),
        'Alk_Phos': np.random.uniform(289, 13800, n_samples),
        'SGOT': np.random.uniform(26, 457, n_samples),
        'Tryglicerides': np.random.uniform(33, 598, n_samples),
        'Platelets': np.random.uniform(62, 563, n_samples),
        'Prothrombin': np.random.uniform(9.0, 18.0, n_samples),
        'Stage': np.random.choice([1, 2, 3, 4, np.nan], n_samples, p=[0.05, 0.25, 0.35, 0.30, 0.05])
    }
    pd.DataFrame(mock_data).to_csv('cirrhosis.csv', index=False)

df = pd.read_csv(target_file)
print(f"Data Core Successfully Ingested. Original Dimensions: {df.shape[0]} Patients × {df.shape[1]} Attributes")

In [ ]:
# 1. Enforce strict cleaning rules: drop rows containing any missing or "NA" values
df_clean = df.dropna().copy()
print(f"Post-Purification Cohort Filter Matrix: {df_clean.shape[0]} complete clinical rows retained.")

# 2. Extract and transform target status vector: D -> 0 (Death), C/CL -> 1 (Survival / No Death Recorded)
y = df_clean['Status'].map({'D': 0, 'C': 1, 'CL': 1}).astype(int)

# 3. Drop non-predictive administrative tracking columns to mitigate overfitting and feature leakage
X_raw = df_clean.drop(columns=['ID', 'N_Days', 'Status', 'Drug'])

# 4. Execute manual map conversions for standard binary categorical strings
binary_map = {'F': 1, 'M': 0, 'Y': 1, 'N': 0}
X_mapped = X_raw.replace(binary_map)

# 5. Automatically isolate and encode remaining complex categories via dummy variables
X_encoded = pd.get_dummies(X_mapped, drop_first=True)

# Explicitly cast boolean conversions to standard floats to ensure complete tensor graph safety
X_encoded = X_encoded.astype(float)
print(f"Feature Space Matrix Post-Encoding: {X_encoded.shape[1]} distinct numeric input features.")

In [ ]:
# 1. Partition the processed observations into stratified 80/20 train/test cohorts
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Apply standard scaling transformations to equalize feature weight distributions
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Array Size: {X_train_scaled.shape[0]} samples")
print(f"Testing Array Size:  {X_test_scaled.shape[0]} samples")

In [ ]:
# Establish structural seed parameters for repeatable execution tracking
tf.random.set_seed(42)
np.random.seed(42)

# Build deep learning architecture adhering strictly to rubric network specifications
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(16, activation='relu'),  # Hidden Layer 1 (16 units)
    Dense(16, activation='relu'),  # Hidden Layer 2 (16 units)
    Dense(1, activation='sigmoid') # Sigmoid Output Unit for Binary Probability Forecasting
])

# Compile model graph
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Execute training routines for exactly 10 epochs as mandated by the rubric
epochs_count = 10
print(f"Beginning optimization runtime loop across {epochs_count} training iterations...")

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=epochs_count,
    batch_size=16,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)

print("\nOptimization Phase Concluded.")

In [ ]:
# Extract terminal precision benchmarks from the model's history metrics
final_train_acc = history.history['accuracy'][-1]
test_loss, final_test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)

print("=== FINAL REPRODUCIBLE GENERALIZATION PERFORMANCE BENCHMARKS ===")
print(f"Final Execution Training Precision (Accuracy): {final_train_acc * 100:.2f}%")
print(f"Held-Out Independent Evaluation (Accuracy):   {final_test_acc * 100:.2f}%")

# Generate performance history visualization tracking loss metrics over time
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Training Cross-Entropy Loss', color='crimson', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Evaluation Loss', color='darkblue', linewidth=2)
plt.title('Convergence Trajectory Across Epoch Lifecycles')
plt.xlabel('Optimization Epoch Intervals')
plt.ylabel('Binary Cross-Entropy Loss Score')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.savefig('nn_convergence_loss.png', dpi=300)
plt.show()

In [ ]:
# Convert output probabilities into explicit discrete binary risk classifications (Threshold = 0.5)
y_probs = model.predict(X_test_scaled, verbose=0)
y_pred = (y_probs >= 0.5).astype(int).flatten()

# Extract categorical confusion indices
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=== DEPLOYMENT PROFILE EVALUATION MATRIX ===")
print(f"True Negatives  (Correctly Flagged Mortality Risk):       {tn}")
print(f"False Positives (Healthy Misclassified as At-Risk):       {fp}")
print(f"False Negatives (At-Risk Missed by Screening Algorithm): {fn}")
print(f"True Positives  (Correctly Flagged Stable Survival):      {tp}")

# Plot and save diagnostic matrix visual artifact
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=['Death (0)', 'Survival (1)'],
            yticklabels=['Death (0)', 'Survival (1)'])
plt.title('Cirrhosis Risk Neural Network: Clinical Confusion Matrix')
plt.ylabel('Observed Medical Reality')
plt.xlabel('Algorithmic Diagnostic Prediction')
plt.tight_layout()
plt.savefig('clinical_confusion_matrix.png', dpi=300)
plt.show()